
# Klasifikasi BMI — Skrining Sleman 2025

Notebook ini mendalami **klasifikasi status gizi berdasarkan Indeks Massa Tubuh (IMT/BMI)**
pada data yang sudah dibersihkan, memakai dua standar yang berbeda menurut kelompok usia.

Seluruh logika klasifikasi berada di modul bersama `bmi_standards.py`, bukan disalin ulang
di sini, supaya definisinya tidak tercecer antar notebook.

## Standar yang dipakai

Sumber tabel: <https://www.calculator.net/bmi-calculator.html>

| Kelompok usia | Standar | Alasan |
|---|---|---|
| **> 20 tahun** | WHO (dewasa), 8 kategori | Tabel WHO dinyatakan berlaku untuk usia 20 tahun ke atas |
| **2 - 20 tahun** | CDC BMI-for-age percentile, per jenis kelamin | IMT anak berubah menurut usia & jenis kelamin, sehingga ambang tetap tidak berlaku |
| **< 2 tahun** | *tidak diklasifikasi* | BMI-for-age tidak berlaku secara medis; WHO merekomendasikan *Weight-for-Length* |

**Catatan batas usia.** Pada sumbernya, tabel WHO dinyatakan berlaku untuk usia ">= 20"
sedangkan tabel CDC untuk "2-20", sehingga **usia 20 tercakup oleh keduanya**. Proyek ini
memilih memasukkan usia 20 ke kelompok anak (CDC), dan batas ini dikendalikan satu
konstanta `bs.CHILD_MAX_AGE` agar mudah diubah bila rujukan PKL menentukan lain.

**Standar Indonesia (Kemenkes/IMT) tidak diimplementasikan** karena tidak ditemukan sumber
yang dapat disitasi pada saat penyusunan. Menuliskan ambang batas medis tanpa rujukan yang
jelas berisiko untuk laporan akademik. Struktur modul memungkinkan penambahannya nanti
tanpa mengubah notebook ini.

In [ ]:

# ===================== IMPORTS & LOAD =====================
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Modul bersama + konfigurasi wilayah. Akar proyek dicari dari folder kerja,
# jadi tidak ada satu pun jalur absolut di notebook ini: seluruh folder boleh
# dipindah atau diserahkan ke orang lain tanpa sel ini perlu diubah.
WILAYAH = "Sleman"

AKAR = Path.cwd()
while not (AKAR / "Bersama").exists():
    AKAR = AKAR.parent
sys.path.insert(0, str(AKAR / "Bersama"))

import konfigurasi as cfg      # jalur + cara baca berkas mentah tiap wilayah
import bmi_standards as bs      # standar BMI & batas plausibilitas per usia
import viz_style as vz          # gaya grafik seragam untuk semua notebook

vz.apply_style()
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 170)

DATA_DIR = cfg.folder_data(WILAYAH)

df = pd.read_csv(os.path.join(DATA_DIR, "dataset_master.csv"))

TARGETS = [
    "Berisiko Diabetes Melitus",
    "Berisiko Hipertensi",
    "Berisiko Kanker Paru",
    "Berisiko TBC",
]

print(f"Sumber standar : {bs.SOURCE_URL}")
print(f"Batas usia anak: {bs.CHILD_MIN_AGE} - {bs.CHILD_MAX_AGE} tahun")
print(f"Data           : {len(df):,} catatan skrining")


## 1. Tabel Rujukan

In [ ]:

# ===================== TABEL WHO (DEWASA) =====================
who = []
prev = 0.0
for upper, label in bs.WHO_ADULT_BINS:
    rentang = f"< {upper:g}" if prev == 0 else (f">= {prev:g}" if np.isinf(upper) else f"{prev:g} - {upper:g}")
    who.append({"Kategori WHO": label, "Rentang IMT (kg/m2)": rentang,
                "Ringkas (WHO-4)": bs.WHO8_TO_WHO4[label]})
    prev = upper

display(pd.DataFrame(who))

In [ ]:

# ===================== TABEL CDC (ANAK 2-20 TAHUN) =====================
print("Kategori CDC berdasarkan persentil BMI-for-age:")
display(pd.DataFrame([
    {"Kategori": bs.CDC_CHILD_LABELS["underweight"], "Rentang persentil": "< 5%"},
    {"Kategori": bs.CDC_CHILD_LABELS["healthy"], "Rentang persentil": "5% - 85%"},
    {"Kategori": bs.CDC_CHILD_LABELS["at_risk"], "Rentang persentil": "85% - 95%"},
    {"Kategori": bs.CDC_CHILD_LABELS["overweight"], "Rentang persentil": "> 95%"},
]))

print("\nTabel ambang persentil per usia & jenis kelamin (1=L, 2=P):")
display(bs.CDC_BMI_REF.pivot(index="Usia", columns="Sex", values=["P5", "P85", "P95"]))


Perhatikan bagaimana ambangnya **bergerak menurut usia**. Untuk anak laki-laki, ambang
overweight (P95) naik dari sekitar 18.8 pada usia 2 tahun ke sekitar 30.6 pada usia 20
tahun. Inilah sebabnya ambang dewasa yang tetap (25 / 30) tidak boleh diterapkan pada
anak — anak usia 8 tahun dengan IMT 21 sudah tergolong overweight menurut CDC, padahal
menurut ambang dewasa ia masih "Normal".


## 2. Hasil Klasifikasi

In [ ]:

# ===================== DISTRIBUSI KATEGORI =====================
# Kolom sudah dihitung di notebook Persiapan Data memakai modul yang sama.
dist = pd.DataFrame({
    "Jumlah": df["Kategori_BMI_WHO8"].value_counts(),
    "%": (df["Kategori_BMI_WHO8"].value_counts(normalize=True) * 100).round(2),
})
display(dist)

# Verifikasi: tidak boleh ada label yang dipakai lintas kelompok usia.
rentang = df.groupby("Kategori_BMI_WHO8")["Usia"].agg(["min", "max", "size"])
rentang.columns = ["Usia min", "Usia maks", "N"]
display(rentang.sort_values("Usia min"))


Tabel rentang usia di atas berfungsi sebagai **uji regresi**: setiap label hanya boleh
muncul pada satu kelompok usia. Label berawalan `Anak_` harus berhenti di usia 20, label
WHO harus mulai dari 21, dan `Balita (0-2th)` harus berhenti di usia 1.

Versi sebelumnya menulis label dewasa dan label anak ke dalam satu kolom **tanpa awalan**,
sehingga `Overweight` berarti dua hal berbeda sekaligus. Akibatnya kolom kategori BMI
diam-diam berfungsi sebagai penanda kelompok usia, dan setiap angka *feature importance*
yang dihitung darinya menjadi tidak sahih.

In [ ]:

# ===================== DISTRIBUSI PER KELOMPOK =====================
dewasa = df[df["Usia"] > bs.CHILD_MAX_AGE].copy()
anak = df[df["Usia"].between(bs.CHILD_MIN_AGE, bs.CHILD_MAX_AGE)].copy()
balita = df[df["Usia"] < bs.CHILD_MIN_AGE].copy()

print(f"Dewasa (> {bs.CHILD_MAX_AGE} th) : {len(dewasa):,}")
print(f"Anak ({bs.CHILD_MIN_AGE}-{bs.CHILD_MAX_AGE} th)      : {len(anak):,}")
print(f"Balita (< {bs.CHILD_MIN_AGE} th)  : {len(balita):,}")

ORDER8 = [l for _, l in bs.WHO_ADULT_BINS]

fig, axes = plt.subplots(1, 2, figsize=vz.FIG_1X2)

d1 = dewasa["Kategori_BMI_WHO8"].value_counts().reindex(ORDER8).fillna(0)
axes[0].bar(range(len(d1)), d1.values, color="steelblue")
axes[0].set_xticks(range(len(d1)))
axes[0].set_xticklabels(d1.index, rotation=35, ha="right", fontsize=8)
axes[0].set_title(f"Kategori WHO — Dewasa (> {bs.CHILD_MAX_AGE} th, n={len(dewasa):,})")
axes[0].set_ylabel("Jumlah")
for i, v in enumerate(d1.values):
    axes[0].text(i, v, f"{int(v):,}", ha="center", va="bottom", fontsize=7)
axes[0].margins(y=0.15)

d2 = anak["Kategori_BMI_WHO8"].value_counts()
axes[1].bar(range(len(d2)), d2.values, color="mediumseagreen")
axes[1].set_xticks(range(len(d2)))
axes[1].set_xticklabels(d2.index, rotation=35, ha="right", fontsize=8)
axes[1].set_title(f"Kategori CDC — Anak ({bs.CHILD_MIN_AGE}-{bs.CHILD_MAX_AGE} th, n={len(anak):,})")
axes[1].set_ylabel("Jumlah")
for i, v in enumerate(d2.values):
    axes[1].text(i, v, f"{int(v):,}", ha="center", va="bottom", fontsize=7)
axes[1].margins(y=0.15)

plt.tight_layout()
plt.show()


## 3. WHO-8 vs WHO-4 — Dampak Peringkasan Kategori

Dataset menyediakan dua tingkat granularitas. Bagian ini memperlihatkan apa yang hilang
saat 8 kategori diringkas menjadi 4.

In [ ]:

# ===================== CROSSTAB WHO8 -> WHO4 =====================
ct = pd.crosstab(dewasa["Kategori_BMI_WHO8"], dewasa["Kategori_BMI_WHO4"])
ct = ct.reindex(index=[c for c in ORDER8 if c in ct.index])
display(ct)

fig, ax = plt.subplots(figsize=vz.FIG_SINGLE)
who4_order = ["Underweight", "Normal", "Overweight", "Obese"]
bottom = np.zeros(len(who4_order))
palette = plt.cm.viridis(np.linspace(0.15, 0.9, len(ORDER8)))

for lab, colr in zip(ORDER8, palette):
    if lab not in ct.index:
        continue
    vals = [ct.loc[lab, c] if c in ct.columns else 0 for c in who4_order]
    ax.bar(who4_order, vals, bottom=bottom, label=lab, color=colr)
    bottom += np.array(vals, dtype=float)

ax.set_title("Komposisi Kategori WHO-4 menurut Kategori WHO-8 (dewasa)")
ax.set_ylabel("Jumlah")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()


Peringkasan menggabungkan tiga tingkat *thinness* menjadi satu "Underweight", dan tiga
kelas obesitas menjadi satu "Obese". Untuk pemodelan, versi ringkas biasanya cukup dan
mengurangi jumlah kolom. Untuk pelaporan kesehatan masyarakat, versi 8 kategori lebih
informatif karena **Obese Class III** menuntut penanganan yang berbeda dari
**Obese Class I**.


## 4. Mengapa Ambang Dewasa Tidak Berlaku untuk Anak

In [ ]:

# ===================== KURVA PERSENTIL CDC vs SEBARAN IMT =====================
fig, axes = plt.subplots(1, 2, figsize=(18, 8), sharey=True)

for ax, (kode, nama) in zip(axes, [(1, "Laki-laki"), (2, "Perempuan")]):
    sub = anak[anak["Jeniskelamin"] == ("L" if kode == 1 else "P")]
    smp = sub.sample(min(12_000, len(sub)), random_state=42)
    ax.scatter(smp["Usia"], smp["IMT_recalculated"], s=3, alpha=0.12, color="grey")

    ref = bs.CDC_BMI_REF[bs.CDC_BMI_REF["Sex"] == kode]
    ax.plot(ref["Usia"], ref["P5"], color="steelblue", lw=2, label="P5 (underweight)")
    ax.plot(ref["Usia"], ref["P85"], color="darkorange", lw=2, label="P85 (at risk)")
    ax.plot(ref["Usia"], ref["P95"], color="crimson", lw=2, label="P95 (overweight)")

    # Ambang dewasa sebagai pembanding
    ax.axhline(25, color="black", ls="--", lw=1.2, label="Ambang dewasa 25")
    ax.axhline(30, color="black", ls=":", lw=1.2, label="Ambang dewasa 30")

    ax.set_title(f"IMT vs Usia — {nama} ({bs.CHILD_MIN_AGE}-{bs.CHILD_MAX_AGE} th)")
    ax.set_xlabel("Usia (tahun)")
    ax.set_ylim(10, 35)
    ax.legend(fontsize=8, loc="upper left")

axes[0].set_ylabel("IMT (kg/m2)")
plt.tight_layout()
plt.show()


Garis persentil CDC **naik seiring usia**, sedangkan ambang dewasa (garis hitam) datar.
Pada usia muda, kurva P95 jauh di bawah 25 — artinya banyak anak yang sudah tergolong
overweight menurut CDC tetapi akan dianggap "Normal" bila memakai ambang dewasa.

In [ ]:

# ===================== SEBERAPA BESAR SALAH KLASIFIKASINYA =====================
# Bandingkan label CDC yang benar dengan label seandainya ambang dewasa dipaksakan.
anak_cmp = anak.copy()
anak_cmp["Jika_pakai_ambang_dewasa"] = anak_cmp["IMT_recalculated"].map(bs.classify_who_adult)
anak_cmp["Kategori_CDC_ringkas"] = anak_cmp["Kategori_BMI_WHO8"].map({
    bs.CDC_CHILD_LABELS["underweight"]: "Underweight",
    bs.CDC_CHILD_LABELS["healthy"]: "Normal",
    bs.CDC_CHILD_LABELS["at_risk"]: "Overweight",
    bs.CDC_CHILD_LABELS["overweight"]: "Overweight",
})
anak_cmp["Dewasa_ringkas"] = anak_cmp["Jika_pakai_ambang_dewasa"].map(bs.collapse_who4)
anak_cmp["Dewasa_ringkas"] = anak_cmp["Dewasa_ringkas"].replace({"Obese": "Overweight"})

beda = anak_cmp["Kategori_CDC_ringkas"] != anak_cmp["Dewasa_ringkas"]
print(f"Anak usia {bs.CHILD_MIN_AGE}-{bs.CHILD_MAX_AGE} th        : {len(anak_cmp):,}")
print(f"Beda klasifikasi bila ambang dewasa dipaksakan: {int(beda.sum()):,} "
      f"({beda.mean()*100:.1f}%)")

display(pd.crosstab(anak_cmp["Kategori_CDC_ringkas"], anak_cmp["Dewasa_ringkas"],
                    rownames=["CDC (benar)"], colnames=["Ambang dewasa (salah)"]))

# Di kelompok usia mana kesalahannya paling parah?
salah_per_usia = (anak_cmp.assign(_b=beda).groupby("Usia")["_b"].mean() * 100).round(1)
fig, ax = plt.subplots(figsize=vz.FIG_WIDE)
ax.bar(salah_per_usia.index, salah_per_usia.values, color="crimson")
ax.set_title("Persentase Anak yang Salah Diklasifikasi bila Memakai Ambang Dewasa")
ax.set_xlabel("Usia (tahun)")
ax.set_ylabel("% salah klasifikasi")
plt.tight_layout()
plt.show()


Kesalahan paling parah terjadi pada **usia muda**, dan menyusut seiring bertambahnya usia —
masuk akal, karena kurva persentil CDC memang mendekati ambang dewasa menjelang usia 20.
Inilah pembenaran kuantitatif untuk memakai dua standar terpisah alih-alih satu ambang
untuk semua umur.


## 5. Kelompok Balita (< 2 Tahun)

Kelompok ini **dilaporkan, bukan disembunyikan**. Mereka tetap ada di dataset dan tetap
dipakai untuk pemodelan TBC, hanya saja kategori BMI-nya sengaja dikosongkan.

In [ ]:

# ===================== PROFIL BALITA =====================
print(f"Jumlah balita (< {bs.CHILD_MIN_AGE} th) : {len(balita):,}")
print(f"Porsi dari seluruh data     : {len(balita)/len(df)*100:.2f}%")
print("(dataset TBC mencakup semua usia, jadi porsinya sama; target lain")
print(" punya batas usia minimal 15 atau 45 th sehingga tidak memuat balita)")
print()
display(balita[["Usia", "Tinggibadan", "Berat Badan", "IMT_recalculated"]].describe().round(2))

# Apakah risiko TBC pada balita cukup berarti untuk dipertahankan?
tbc_balita = (balita["Berisiko TBC"] == "Berisiko").mean() * 100
tbc_anak = (df[df["Usia"].between(2, 14)]["Berisiko TBC"] == "Berisiko").mean() * 100
tbc_semua = (df["Berisiko TBC"] == "Berisiko").mean() * 100
print(f"\nPrevalensi risiko TBC:")
print(f"  Balita (< 2 th) : {tbc_balita:.2f}%  (n={len(balita):,})")
print(f"  Anak (2-14 th)  : {tbc_anak:.2f}%")
print(f"  Seluruh populasi: {tbc_semua:.2f}%")


### Mengapa balita tetap dipertahankan tanpa kategori BMI

Tinggi dan berat mereka wajar sebagai antropometri bayi (median sekitar 71 cm / 8 kg), dan
prevalensi risiko TBC-nya **sebanding dengan kelompok anak yang lebih tua**. Jadi mereka
adalah kelompok yang nyata dan relevan secara klinis — TBC pada bayi justru berbahaya.

Yang tidak tersedia hanyalah **kategori BMI-nya**, karena BMI-for-age memang tidak berlaku
di bawah usia 2 tahun. Ada tiga pilihan, dan hanya satu yang bisa dipertanggungjawabkan:

| Pilihan | Penilaian |
|---|---|
| Buang seluruh barisnya | Membuang ratusan bayi berisiko TBC, dan membuat model TBC diam-diam tidak berlaku untuk kelompok termuda |
| Paksakan ke kategori usia 2 tahun | Mengarang klasifikasi klinis yang standarnya sendiri menyatakan tidak berlaku |
| **Pertahankan, kosongkan kategorinya** | **Dipilih.** Baris tetap menyumbang usia, jenis kelamin, tinggi, berat, dan segmen; hanya fitur turunan yang absen |

Kehilangan kategori BMI untuk kelompok ini praktis tidak merugikan model, karena kategori
BMI memang redundan terhadap `IMT_recalculated` yang tetap tersedia.


## 6. Profil Status Gizi Populasi

In [ ]:

# ===================== STATUS GIZI DEWASA per SUBKELOMPOK =====================
AGE_BINS = [21, 30, 40, 50, 60, 70, 200]
AGE_LABELS = ["21-29", "30-39", "40-49", "50-59", "60-69", "70+"]
dewasa = dewasa.assign(
    Kelompok_Usia=pd.cut(dewasa["Usia"], bins=AGE_BINS, labels=AGE_LABELS, right=False)
)
WHO4_ORDER = ["Underweight", "Normal", "Overweight", "Obese"]

fig, axes = plt.subplots(1, 3, figsize=(20, 7.5))

for ax, kolom, judul in [
    (axes[0], "Jeniskelamin", "Jenis Kelamin"),
    (axes[1], "Kelompok_Usia", "Kelompok Usia"),
    (axes[2], "segmen", "Segmen Kepesertaan"),
]:
    comp = (pd.crosstab(dewasa[kolom], dewasa["Kategori_BMI_WHO4"], normalize="index") * 100)
    comp = comp.reindex(columns=WHO4_ORDER).fillna(0)
    comp.plot(kind="bar", stacked=True, ax=ax,
              color=["#6BAED6", "#74C476", "#FD8D3C", "#DE2D26"])
    ax.set_title(f"Status Gizi menurut {judul}")
    ax.set_ylabel("% dalam kelompok")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=35)
    ax.legend(fontsize=7, loc="lower right")

plt.tight_layout()
plt.show()

In [ ]:

# ===================== BEBAN GIZI GANDA =====================
gizi = (dewasa["Kategori_BMI_WHO4"].value_counts(normalize=True) * 100).reindex(WHO4_ORDER)
print("Status gizi dewasa (> 20 th):")
for k, v in gizi.items():
    print(f"  {k:<12} {v:>6.2f}%")
print()
print(f"Kekurangan gizi (Underweight)       : {gizi['Underweight']:.2f}%")
print(f"Kelebihan gizi (Overweight + Obese) : {gizi[['Overweight', 'Obese']].sum():.2f}%")

# Beban ganda per segmen: apakah keduanya tinggi di kelompok yang sama?
beban = pd.DataFrame({
    "Underweight (%)": (dewasa.groupby("segmen")["Kategori_BMI_WHO4"]
                        .apply(lambda s: (s == "Underweight").mean() * 100)).round(2),
    "Overweight+Obese (%)": (dewasa.groupby("segmen")["Kategori_BMI_WHO4"]
                             .apply(lambda s: s.isin(["Overweight", "Obese"]).mean() * 100)).round(2),
})
beban["Total beban (%)"] = (beban.sum(axis=1)).round(2)
display(beban.sort_values("Total beban (%)", ascending=False))


**Beban gizi ganda (*double burden of malnutrition*).** Kekurangan gizi dan kelebihan gizi
hadir bersamaan dalam populasi yang sama, bahkan dalam segmen kepesertaan yang sama. Ini
temuan kesehatan masyarakat yang berdiri sendiri, terlepas dari model prediksi apa pun,
dan berimplikasi bahwa program intervensi gizi tidak bisa hanya menyasar satu arah.


## 7. Kategori BMI vs Risiko Penyakit

In [ ]:

# ===================== PREVALENSI RISIKO per KATEGORI BMI =====================
# Dibatasi ke dewasa supaya ambang yang dipakai konsisten.
hasil = {}
for col in TARGETS:
    sub = dewasa[dewasa[col].isin(["Berisiko", "Tidak Berisiko"])]
    if sub.empty:
        continue
    hasil[col.replace("Berisiko ", "")] = (
        sub.assign(_p=sub[col].eq("Berisiko"))
           .groupby("Kategori_BMI_WHO8")["_p"].mean() * 100
    ).round(2)

tab = pd.DataFrame(hasil).reindex([c for c in ORDER8])
display(tab)

fig, axes = plt.subplots(2, 2, figsize=vz.FIG_2X2)
for ax, col in zip(axes.flat, tab.columns):
    vals = tab[col].dropna()
    ax.barh(range(len(vals)), vals.values, color="steelblue")
    ax.set_yticks(range(len(vals)))
    ax.set_yticklabels(vals.index, fontsize=8)
    ax.invert_yaxis()
    ax.set_title(col)
    ax.set_xlabel("% Berisiko")
    for i, v in enumerate(vals.values):
        ax.text(v, i, f" {v:.0f}", va="center", fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:

# ===================== APAKAH LABELNYA SEKADAR ATURAN IMT? =====================
rows = []
for col in TARGETS:
    sub = dewasa[dewasa[col].isin(["Berisiko", "Tidak Berisiko"])]
    if sub.empty:
        continue
    pos = sub[col].eq("Berisiko")
    hi = sub["IMT_recalculated"] > 25
    rows.append({
        "Target": col.replace("Berisiko ", ""),
        "N dewasa": f"{len(sub):,}",
        "P(Berisiko | IMT > 25)": round(pos[hi].mean(), 4),
        "P(Berisiko | IMT <= 25)": round(pos[~hi].mean(), 4),
        "Pengecualian di IMT > 25": int((~pos[hi]).sum()),
        "Selisih": round(pos[hi].mean() - pos[~hi].mean(), 4),
    })

display(pd.DataFrame(rows))


### Temuan — label Diabetes praktis merupakan aturan `IMT > 25`

Grafik dan tabel di atas memperlihatkan **fungsi tangga**, bukan gradien: pada responden
dewasa, hampir setiap orang ber-IMT di atas 25 diberi label `Berisiko Diabetes Melitus`,
dengan jumlah pengecualian yang sangat kecil.

Hipertensi menunjukkan hubungan kuat tetapi **tidak** deterministik. Kanker Paru dan TBC
praktis tidak terpengaruh IMT sama sekali — selisih prevalensinya mendekati nol.

**Konsekuensi yang harus dibawa ke laporan:**

1. **Prevalensi "berisiko diabetes" bukan temuan epidemiologis yang berdiri sendiri.**
   Sebagian besar hanyalah pernyataan ulang dari prevalensi overweight pada populasi ini.
   Melaporkannya sebagai ukuran risiko diabetes yang independen akan menyesatkan.
2. **Model diabetes yang memakai IMT sebagai fitur sedang merekonstruksi aturan penilaian
   kuesioner**, bukan memprediksi penyakit. Akurasi tinggi pada target ini tidak boleh
   dibaca sebagai keberhasilan prediksi.
3. Untuk Kanker Paru dan TBC, kategori BMI **hampir tidak membawa informasi**. Faktor yang
   relevan di sana adalah jenis kelamin (Kanker Paru) serta usia dan segmen (TBC).


## 8. Ringkasan

In [ ]:

# ===================== RINGKASAN =====================
print("=" * 72)
print("RINGKASAN KLASIFIKASI BMI")
print("=" * 72)
print(f"Sumber standar : {bs.SOURCE_URL}")
print()
print(f"Dewasa (> {bs.CHILD_MAX_AGE} th)  : {len(dewasa):>7,}  -> WHO, 8 kategori")
print(f"Anak ({bs.CHILD_MIN_AGE}-{bs.CHILD_MAX_AGE} th)       : {len(anak):>7,}  -> CDC BMI-for-age percentile")
print(f"Balita (< {bs.CHILD_MIN_AGE} th)   : {len(balita):>7,}  -> tidak diklasifikasi")
print(f"{'TOTAL':<20} {len(df):>7,}")
print()
print("Kategori yang dihasilkan:")
print(f"  Kategori_BMI_WHO8 : {df['Kategori_BMI_WHO8'].nunique()} nilai unik")
print(f"  Kategori_BMI_WHO4 : {df['Kategori_BMI_WHO4'].nunique()} nilai unik")


### Catatan penutup

1. **Dua standar dipakai karena satu ambang tidak berlaku untuk semua umur.** IMT anak
   berubah menurut usia dan jenis kelamin; memaksakan ambang dewasa akan salah
   mengklasifikasikan sebagian besar anak usia muda (lihat Bagian 4).

2. **Usia di bawah 2 tahun sengaja tidak diklasifikasi.** WHO merekomendasikan
   *Weight-for-Length* untuk kelompok ini, dan tabel rujukannya tidak tersedia dengan
   sumber yang dapat disitasi. Barisnya tetap dipertahankan karena relevan untuk TBC.

3. **Standar Indonesia (Kemenkes/IMT) tidak diimplementasikan** karena tidak ditemukan
   sumber yang dapat disitasi pada saat penyusunan. Bila rujukan resminya ditemukan,
   penambahannya cukup dilakukan di `bmi_standards.py` tanpa mengubah notebook ini.

4. **Kategori BMI bersifat redundan terhadap `IMT_recalculated`** — keduanya membawa
   informasi yang sama, hanya beda bentuk. Notebook modeling menguji hal ini secara
   eksplisit lewat ablasi fitur.